
# 🤖 AI Software Engineer Assistant

A portfolio-level GenAI project that acts as an AI software engineer.

### What this notebook demonstrates
- LLM-powered software requirement analysis
- Repository/code understanding
- Code generation
- Unit-test generation
- Bug diagnosis and debugging suggestions
- Code review
- Tool-style agent workflow
- Structured JSON outputs
- LangGraph-style orchestration using Python
- OpenRouter-compatible LLM integration

> **Goal:** Build the core AI engineering workflow in one notebook first. The same components can later be moved into a FastAPI + React production application.



## 🏗️ Architecture

```text
                         User Requirement
                                │
                                ▼
                       ┌─────────────────┐
                       │   Planner LLM   │
                       └────────┬────────┘
                                │
             ┌──────────────────┼──────────────────┐
             ▼                  ▼                  ▼
       Code Analyzer       Code Generator      Test Generator
             │                  │                  │
             └──────────────────┼──────────────────┘
                                ▼
                         Debugger / Reviewer
                                │
                                ▼
                         Final Engineer Report

Optional tools:
  • Repository/file reader
  • Python execution
  • Test runner
  • Git/GitHub integration
  • Documentation search



## 1. Project Setup

### Install dependencies

Run this cell once in a fresh environment.


In [ ]:

%pip install -q openai python-dotenv pydantic



## 2. Environment Configuration

Create a `.env` file:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

The notebook uses OpenRouter's OpenAI-compatible API.


In [ ]:

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

if not OPENROUTER_API_KEY:
    print("⚠️ Set OPENROUTER_API_KEY in your .env file before calling the LLM.")
else:
    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
    )
    print(f"✅ Client configured. Model: {MODEL}")


## 3. Core LLM Helper

In [ ]:

from typing import Optional

def ask_llm(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.2,
) -> str:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY is not configured.")

    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


## 4. Software Requirement Analyzer

In [ ]:

def analyze_requirement(requirement: str) -> str:
    system = """
You are a senior software architect and AI software engineer.
Analyze software requirements precisely.

Return:
1. Functional requirements
2. Non-functional requirements
3. Assumptions
4. Suggested architecture
5. Components/modules
6. API requirements
7. Data model
8. Testing strategy
9. Risks
10. Implementation plan

Do not invent requirements that are not supported by the request.
"""

    return ask_llm(system, requirement)

requirement = """
Build a REST API for a job application portal.
Users should create accounts, store resumes, track applications,
and receive AI-generated job matching recommendations.
Use Python FastAPI and MongoDB.
"""

print(analyze_requirement(requirement))


## 5. Code Analyzer

In [ ]:

def analyze_code(code: str, language: str = "python") -> str:
    system = f"""
You are an expert {language} code analyst.

Analyze the supplied code for:
- Architecture
- Correctness
- Bugs
- Security problems
- Performance problems
- Maintainability
- Error handling
- Code smells
- Missing tests

Give concrete findings and explain why each matters.
"""

    return ask_llm(
        system,
        f"Analyze this {language} code:\n\n```{language}\n{code}\n```"
    )

sample_code = """
def calculate_average(numbers):
    total = 0
    for number in numbers:
        total += number
    return total / len(numbers)

print(calculate_average([]))
"""

print(analyze_code(sample_code))


## 6. AI Code Generator

In [ ]:

def generate_code(task: str, language: str = "python") -> str:
    system = f"""
You are a senior {language} software engineer.

Generate production-oriented code for the requested task.

Requirements:
- Use clear structure
- Add type hints where appropriate
- Handle errors
- Avoid unnecessary dependencies
- Include concise comments
- Return only the implementation unless explanation is required
"""

    return ask_llm(
        system,
        f"Implement the following requirement in {language}:\n\n{task}",
        temperature=0.1,
    )

task = """
Create a Python function that validates an email address,
rejects empty input, and returns a structured result containing
is_valid and reason.
"""

generated_code = generate_code(task)
print(generated_code)


## 7. AI Unit Test Generator

In [ ]:

def generate_tests(code: str, language: str = "python") -> str:
    system = f"""
You are a senior test engineer.

Generate comprehensive tests for the supplied {language} code.

Cover:
- Happy paths
- Edge cases
- Invalid inputs
- Boundary conditions
- Exceptions
- Regression cases

Prefer pytest for Python.
Return runnable test code.
"""

    return ask_llm(
        system,
        f"Generate tests for:\n\n```{language}\n{code}\n```",
        temperature=0.1,
    )

print(generate_tests(generated_code))


## 8. AI Debugger

In [ ]:

def debug_code(code: str, error: str, language: str = "python") -> str:
    system = f"""
You are an expert debugging engineer.

Given source code and an observed error:
1. Identify the root cause.
2. Explain the failure.
3. Propose the smallest safe fix.
4. Provide corrected code.
5. Suggest a regression test.

Do not hide uncertainty.
"""

    prompt = f"""
Language: {language}

SOURCE CODE:
```{language}
{code}
```

ERROR:
{error}
"""

    return ask_llm(system, prompt, temperature=0.1)

buggy_code = """
def divide(a, b):
    return a / b

result = divide(10, 0)
"""

error = "ZeroDivisionError: division by zero"

print(debug_code(buggy_code, error))


## 9. AI Code Reviewer

In [ ]:

def review_code(code: str, language: str = "python") -> str:
    system = f"""
You are a principal software engineer performing a code review.

Review the code for:
- Correctness
- Security
- Performance
- Maintainability
- Readability
- API design
- Error handling
- Testing
- Production readiness

For every issue provide:
Severity | Finding | Why | Recommended fix

Then provide a concise review summary.
"""

    return ask_llm(
        system,
        f"Review this {language} code:\n\n```{language}\n{code}\n```",
        temperature=0.1,
    )

print(review_code(sample_code))



## 10. Repository Understanding

For a real project, collect source files and provide their contents to the analyzer.

The helper below intentionally skips common generated/binary directories.

> For large repositories, do not send the entire repository to the LLM at once. Use file selection, summaries, embeddings, or a repository index.


In [ ]:

from pathlib import Path

IGNORED_DIRS = {
    ".git", ".venv", "venv", "node_modules",
    "__pycache__", ".next", "dist", "build"
}

SUPPORTED_EXTENSIONS = {
    ".py", ".js", ".jsx", ".ts", ".tsx",
    ".java", ".go", ".rs", ".sql", ".md"
}

def collect_repository_files(root: str, max_chars_per_file: int = 12000):
    root_path = Path(root)
    files = []

    for path in root_path.rglob("*"):
        if not path.is_file():
            continue

        if any(part in IGNORED_DIRS for part in path.parts):
            continue

        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        files.append({
            "path": str(path.relative_to(root_path)),
            "content": content[:max_chars_per_file],
        })

    return files

# Example:
# repository_files = collect_repository_files("./my_project")
# print(f"Collected {len(repository_files)} source files.")


## 11. Repository-Level Architecture Analysis

In [ ]:

def analyze_repository(repository_files) -> str:
    if not repository_files:
        return "No repository files were supplied."

    combined = "\n\n".join(
        f"FILE: {item['path']}\n{item['content']}"
        for item in repository_files
    )

    # Keep the request bounded for a notebook demonstration.
    combined = combined[:60000]

    system = """
You are a principal software architect.

Analyze a software repository from the supplied source files.

Produce:
- Project purpose
- Architecture
- Important modules
- Dependency relationships
- API/data flow
- Potential bugs
- Security concerns
- Technical debt
- Testing gaps
- Recommended improvements

Clearly distinguish observed facts from inferred architecture.
"""

    return ask_llm(system, combined, temperature=0.1)

# Example:
# repository_files = collect_repository_files("./my_project")
# print(analyze_repository(repository_files))



## 12. Agentic Software Engineer Workflow

This section combines the individual capabilities into an engineer-style workflow.

```text
Requirement
     ↓
Planner
     ↓
 ┌───┼────────────┐
 ↓   ↓            ↓
Code Test       Review
 ↓   ↓            ↓
 └───┼────────────┘
     ↓
Debugger
     ↓
Final Engineering Report
```

This notebook keeps orchestration explicit so each step can be inspected and later migrated to LangGraph.


In [ ]:

def software_engineer_assistant(
    requirement: str,
    language: str = "python",
) -> dict:
    plan = analyze_requirement(requirement)

    implementation = generate_code(
        task=requirement,
        language=language,
    )

    tests = generate_tests(
        code=implementation,
        language=language,
    )

    review = review_code(
        code=implementation,
        language=language,
    )

    return {
        "plan": plan,
        "implementation": implementation,
        "tests": tests,
        "review": review,
    }

# Example:
# result = software_engineer_assistant(
#     "Create a Python REST API endpoint that validates a job application."
# )
# print(result["plan"])
# print(result["implementation"])
# print(result["tests"])
# print(result["review"])


## 13. Optional: LangGraph Upgrade Path


For a production version, replace the sequential Python orchestration with **LangGraph**.

Suggested graph:

```text
START
  ↓
Planner
  ↓
Repository Analyzer
  ↓
Implementation
  ↓
Test Generator
  ↓
Test Executor
  ↓
┌───────────────┐
│ Tests passed? │
└───────┬───────┘
    Yes │ No
        │
        ├──────────────→ Debugger
        │                    │
        │                    └──→ Test Executor
        ↓
Code Reviewer
  ↓
Security Reviewer
  ↓
Documentation Agent
  ↓
END
```

This introduces genuine **agentic iteration** instead of simply chaining LLM prompts.


## 14. Production Extensions


### 🔥 Next-level features

1. **GitHub integration**
   - Read repositories
   - Create branches
   - Analyze issues
   - Generate pull requests

2. **Code execution sandbox**
   - Run generated code safely
   - Capture stdout/stderr
   - Feed failures back to the debugger

3. **Automated test loop**
   - Generate → execute → debug → retest

4. **Repository RAG**
   - Chunk source files
   - Generate embeddings
   - Store in vector database
   - Retrieve relevant code before generation

5. **MCP**
   - GitHub MCP
   - Filesystem MCP
   - Database MCP
   - Documentation/search MCP

6. **Memory**
   - Project architecture memory
   - Developer preferences
   - Previous bugs/fixes
   - Previous architectural decisions

7. **Guardrails**
   - Prevent destructive commands
   - Validate generated code
   - Detect secrets
   - Restrict filesystem access

8. **Observability**
   - LangSmith traces
   - Token usage
   - Latency
   - Agent steps
   - Failure rates

9. **Evaluation**
   - Code correctness
   - Test pass rate
   - Security findings
   - Hallucination rate
   - Patch success rate

10. **Production UI**
    - React/Next.js frontend
    - FastAPI backend
    - Streaming responses
    - Repository upload/connect
    - Agent execution timeline



## 🎯 Portfolio Outcome

After completing this notebook, the project can evolve into:

**AI Software Engineer Agent**

```text
React / Next.js
       │
       ▼
FastAPI
       │
       ▼
LangGraph Supervisor
       │
 ┌─────┼──────────────┐
 ▼     ▼              ▼
RAG   Coding Tools   MCP Tools
 │     │              │
 ▼     ▼              ▼
Codebase  Sandbox   GitHub/DB/Docs
       │
       ▼
LLM / OpenRouter
       │
       ▼
Tests → Debug → Review → Final Patch
```

This is substantially closer to a real **AI Engineer / Agentic AI project** than a basic chatbot or simple RAG demo.
